In [8]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [9]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    784 non-null    object
 1   label   784 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 12.4+ KB


In [10]:
test['label'] = test['label'].apply(lambda x: 'irony' if x == 1 else 'no irony')

labels = test['label'].unique()

test

,text,label
0,@user Can U Help?||More conservatives needed o...,no irony
1,"Just walked in to #Starbucks and asked for a ""...",irony
2,#NOT GONNA WIN,no irony
3,@user He is exactly that sort of person. Weirdo!,no irony
4,So much #sarcasm at work mate 10/10 #boring 10...,irony
...,...,...
779,"If you drag yesterday into today, your tomorro...",no irony
780,Congrats to my fav @user & her team & my birth...,no irony
781,@user Jessica sheds tears at her fan signing e...,no irony
782,#Irony: al jazeera is pro Anti - #GamerGate be...,irony


In [11]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'no irony' in text:
        return'no irony'
    elif 'irony' in text:
        return 'irony'
    else:
        return 'error'

In [13]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_binary3.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_binary3.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,@user Can U Help?||More conservatives needed o...,no irony,no irony,1.879873,3.0,118.0,121.0,no irony
1,"Just walked in to #Starbucks and asked for a ""...",irony,irony,0.573521,3.0,110.0,113.0,irony
2,#NOT GONNA WIN,no irony,irony,0.655389,3.0,94.0,97.0,irony
3,@user He is exactly that sort of person. Weirdo!,no irony,irony,0.479765,3.0,102.0,105.0,irony
4,So much #sarcasm at work mate 10/10 #boring 10...,irony,irony,0.623052,3.0,126.0,129.0,irony
...,...,...,...,...,...,...,...,...
779,"If you drag yesterday into today, your tomorro...",no irony,no irony,0.474261,3.0,117.0,120.0,no irony
780,Congrats to my fav @user & her team & my birth...,no irony,no irony,0.583564,3.0,124.0,127.0,no irony
781,@user Jessica sheds tears at her fan signing e...,no irony,no irony,0.484773,3.0,106.0,109.0,no irony
782,#Irony: al jazeera is pro Anti - #GamerGate be...,irony,irony,0.496005,3.0,109.0,112.0,irony


In [14]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,@user Can U Help?||More conservatives needed o...,no irony,no irony,1.879873,3.0,118.0,121.0,no irony
1,"Just walked in to #Starbucks and asked for a ""...",irony,irony,0.573521,3.0,110.0,113.0,irony
2,#NOT GONNA WIN,no irony,irony,0.655389,3.0,94.0,97.0,irony
3,@user He is exactly that sort of person. Weirdo!,no irony,irony,0.479765,3.0,102.0,105.0,irony
4,So much #sarcasm at work mate 10/10 #boring 10...,irony,irony,0.623052,3.0,126.0,129.0,irony
...,...,...,...,...,...,...,...,...
779,"If you drag yesterday into today, your tomorro...",no irony,no irony,0.474261,3.0,117.0,120.0,no irony
780,Congrats to my fav @user & her team & my birth...,no irony,no irony,0.583564,3.0,124.0,127.0,no irony
781,@user Jessica sheds tears at her fan signing e...,no irony,no irony,0.484773,3.0,106.0,109.0,no irony
782,#Irony: al jazeera is pro Anti - #GamerGate be...,irony,irony,0.496005,3.0,109.0,112.0,irony


In [15]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.741071
F1 score: 0.738547
Precision: 0.833386
Recall: 0.741071


In [16]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.6500103282076972
Average completion tokens: 3.0
Average prompt tokens: 110.27551020408163
Average total tokens: 113.27551020408163


In [17]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.01437960000000002


In [18]:
with open('results/openai_ZS_binary3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')